# 🧪 Halloumi Hallucination Scoring (standalone)

Runs **after** the main `AIO_WebsiteScrape` notebook, on a **GPU runtime**
(*Runtime ▸ Change runtime type ▸ T4*).

### How to run (free T4)
1. Run the **Install** cell below.
2. **Runtime ▸ Restart session** (required — Colab ships a newer `transformers`
   that breaks 4-bit; the restart loads the pinned versions).
3. Run the **Scoring** cell.

### What it does
- **Statement-level** where the AIO exposed inline citations: each supporting
  statement is scored against the body text of the sources it cites →
  `supporting_statement{N}_halloumi_supported / _unsupported / _n_claims`.
- **Fallback** when a question has no inline citations: the whole `aio_text` is
  scored against all that question's source texts → `aio_halloumi_*` on each row.

### If you hit `CUDA out of memory`
That's **GPU memory**, not the Drive mount. The scoring cell frees prior models
and reports VRAM + whether 4-bit engaged. If it still OOMs, do **Runtime ▸
Restart session** (fully clears the GPU) and run the scoring cell once. The model
must load in 4-bit (~5–6 GB); if the cell prints `4-bit loaded: False`, the
pinned versions need adjusting.

In [ ]:
# @title 📦 Install pinned deps — run, then RESTART the runtime { display-mode: "form" }
# @markdown Pins a tested transformers / bitsandbytes / accelerate combo so 4-bit
# @markdown loading works on a T4 (the latest transformers regresses bnb 4-bit).
# @markdown **After this finishes: Runtime ▸ Restart session, then run the scoring cell.**
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers==4.55.4", "accelerate==1.6.0", "bitsandbytes==0.45.5",
     "tqdm"],
    check=True,
)
# The HallOumi repo is Xet-backed, and HF's Xet CDN intermittently 403s on Colab
# ("Auth failed: SignatureError: invalid key pair id"), which kills the model
# download partway (a single shard fails while the others succeed). The
# HF_HUB_DISABLE_XET env var doesn't reliably take effect because huggingface_hub
# caches it at import time (often before our cell runs). So remove the hf_xet
# package outright: without it, huggingface_hub uses the classic HTTPS resolve
# endpoint and the Xet CDN is never touched. (No-op if it isn't installed.)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "hf_xet"])
print("✅ Installed pinned versions; removed hf_xet so downloads bypass the Xet CDN.")
print("➡️  Now do: Runtime ▸ Restart session, then run the Scoring cell below.")

In [ ]:
# @title 🧪 Halloumi Scoring (run AFTER restarting the runtime) { display-mode: "form" }
# @markdown **Every statement in the AI Overview is enumerated and accounted for.**
# @markdown The AIO text is split into statements; each inherits Google's explicit
# @markdown citations where the AIO metadata provided them. Verdicts per statement:
# @markdown   • **supported**   — a readable cited source clears the threshold
# @markdown   • **unsupported** — ALL cited sources readable, none support it
# @markdown   • **skipped**     — indeterminate: readable sources don't support it
# @markdown     and ≥1 cited source's text couldn't be recovered
# @markdown   • **uncited**     — Google attached no citations (not scored)
# @markdown
# @markdown Outputs (all in `PROJECT_DIR/Output/`):
# @markdown   • `statement_scores.csv` — one row per statement, every statement
# @markdown   • `aio_dataset.csv` enriched in place + per-question totals stamped
# @markdown     on every row (`n_total / n_supported / n_scored / n_skipped /
# @markdown     n_uncited`, `support_score` like "6/8")
# @markdown   • comparison CSVs + **the Excel workbook** (built here, at the end,
# @markdown     so the AIO sheet always carries the halloumi columns)
# @markdown **GPU runtime required.** Needs `comparison_builder.py` in PROJECT_DIR.

PROJECT_DIR    = r"C:/Users/SAgno/Documents/AIO_Workspace/LocalRun"  # @param {type:"string"}
OUTPUT_SUBDIR  = "Output"            # @param {type:"string"}
AIO_FILE       = "aio_dataset.csv"   # @param {type:"string"}
STMT_FILE      = "statement_scores.csv"  # @param {type:"string"}
MAX_STATEMENTS = 8                    # @param {type:"integer"}
LOAD_IN_4BIT   = True                 # @param {type:"boolean"}
# A SENTENCE counts as supported when ANY chunk of ANY cited source scores >= this.
SUPPORT_THRESHOLD = 0.5               # @param {type:"number"}
# A STATEMENT (paragraph) counts as supported when at least this FRACTION of its
# factual sentences are supported. 0.5 = majority; 1.0 = every sentence; lower it
# to be more lenient (e.g. 0.34 = at least a third).
SUPPORT_FRACTION  = 0.5               # @param {type:"number"}
# Chunk window per forward pass. 2800 fits a T4 alongside the 4-bit 8B model.
CHUNK_TOKENS   = 2800                 # @param {type:"integer"}

# --- 0. GPU memory hygiene (set BEFORE importing torch) ---
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Download tuning (set BEFORE huggingface_hub is imported):
USE_HF_TRANSFER = False  # @param {type:"boolean"}
# ^ Rust parallel downloader. Kept OFF by default: this model is Xet-backed, and
#   the plain resumable downloader is the reliable path on Colab. See the
#   HF_HUB_DISABLE_XET note below for why the download otherwise freezes mid-shard.

# A dead socket with no data must TIME OUT (default is effectively forever, which
# is why the download hangs). 60s + the resumable retry loop below = a stalled
# shard errors and resumes instead of hanging.
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"

# The HallOumi repo is Xet-backed. When hf_xet is present, huggingface_hub routes
# downloads through the Xet protocol, which stalls mid-shard on Colab (progress
# freezes at a fixed %, e.g. stuck at 11%). Disabling Xet forces the plain,
# resumable HTTP downloader — slightly slower but it does not hang and resumes.
os.environ["HF_HUB_DISABLE_XET"] = "1"

import subprocess, sys
if USE_HF_TRANSFER:
    try:
        import hf_transfer  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hf_transfer"])
    try:
        import hf_transfer  # noqa: F401
        os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
        print("✓ hf_transfer enabled (set USE_HF_TRANSFER=False if it stalls).")
    except ImportError:
        os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
        print("⚠ hf_transfer unavailable — using the standard downloader.")
else:
    os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
    print("ℹ hf_transfer disabled — using the standard downloader.")

# Optional: use an HF token from Colab secrets (Colab ▸ 🔑). NOT required for
# this public model and does NOT speed downloads up — it only removes the
# HF_TOKEN warning and slightly improves CDN rate limits.
# HF_TOKEN is optional for this public model; read it from the environment
# if the user has one set.
if os.environ.get("HF_TOKEN"):
    print("✓ HF token found in environment.")

import gc
for _v in ("halloumi", "model"):          # free a model left over from a prior run
    if _v in globals():
        del globals()[_v]
import torch
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    _free, _total = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name(0)} | free {_free/1e9:.2f}/{_total/1e9:.2f} GB")
else:
    # Fail FAST — the 4-bit model load needs CUDA and would crash minutes later
    # with a confusing bitsandbytes error on a CPU runtime.
    raise RuntimeError(
        "No GPU detected — this cell needs a GPU runtime.\n"
        "  1. Runtime ▸ Change runtime type ▸ T4 GPU ▸ Save\n"
        "  2. Re-run the Install cell above\n"
        "  3. Runtime ▸ Restart session\n"
        "  4. Re-run this cell"
    )

# --- 1. Local paths (no Drive mount needed) ---

import sys, csv, re, importlib
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)
import pandas as pd
from tqdm.auto import tqdm
from halloumi_scorer import HalloumiScorer

OUT_DIR = os.path.join(PROJECT_DIR, OUTPUT_SUBDIR)

# ── Pause / resume ────────────────────────────────────────────────────────────
# Every scored statement is appended to CKPT_FILE, so a run can stop at any time
# (pause flag, max-hours limit, killed process, reboot) and the next run skips
# everything already done. Create STOP_FLAG to pause cleanly after the current
# statement; HALLOUMI_MAX_HOURS=<n> pauses on its own after n hours.
# halloumi_ctl.py (start | pause | status) wraps all of this.
import atexit, hashlib, json as _json
CKPT_FILE = os.path.join(OUT_DIR, "halloumi_checkpoint.jsonl")
STOP_FLAG = os.path.join(PROJECT_DIR, "logs", "HALLOUMI_STOP")
PID_FILE  = os.path.join(PROJECT_DIR, "logs", "halloumi.pid")
MAX_HOURS = float(os.environ.get("HALLOUMI_MAX_HOURS", "0") or 0)
os.makedirs(os.path.dirname(PID_FILE), exist_ok=True)
open(PID_FILE, "w").write(str(os.getpid()))
atexit.register(lambda: os.path.exists(PID_FILE) and os.remove(PID_FILE))

def _field(val):
    if val is None:
        return ""
    s = str(val).strip()
    return "" if s.lower() in ("", "nan", "none", "nat", "null") else s

def _src_key(val):
    s = _field(val)
    if not s:
        return None
    try:
        return int(float(s))
    except ValueError:
        return None

# Local text-quality classifier — used when the dataset predates the
# text_quality column (mirrors the scraper's assess_text_quality).
_GARBAGE_RE = re.compile(
    r"(checking your browser|just a moment|access denied|error 40\d|"
    r"\bforbidden\b|performing security verification|"
    r"not automatically redirected|please switch to a different browser|"
    r"enable javascript and cookies|requiring captcha|complete the captcha|"
    r"verify you are (?:a human|not a bot)|attention required|error code:|"
    r"security service to protect against malicious bots)",
    re.I)

# Google Shopping product-listing stubs (google.com/search?q=product&prds=...)
# scrape as ~1KB of boilerplate — not real source text. Treat as unusable so
# statements citing them go skipped-indeterminate instead of unsupported.
_PRODUCT_STUB_RE = re.compile(r"google\.[a-z.]+/search|[?&]prds=", re.I)

def _assess_quality(text):
    if text is None or not str(text).strip():
        return "empty"
    t = str(text).strip()
    if len(t) < 2500 and _GARBAGE_RE.search(t[:1500]):
        return "garbage_interstitial"
    if len(t) < 300:
        return "too_short"
    return "ok"

# --- 2. Load (in-place target lives in Output/) ---
path = os.path.join(OUT_DIR, AIO_FILE)
df = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
print(f"Loaded {len(df)} AIO rows from {OUTPUT_SUBDIR}/{AIO_FILE}.")
if "source_index" not in df.columns:
    raise KeyError("source_index column missing — re-run Step 9 (it now emits source_index).")

# source_index -> {"text", "quality"} per question.
src_by_q = {}
for q, grp in df.groupby("question"):
    m = {}
    for _, r in grp.iterrows():
        k = _src_key(r.get("source_index"))
        if k is None:
            continue
        text = _field(r.get("text"))
        quality = _field(r.get("text_quality")) or _assess_quality(text)
        if quality == "ok" and _field(r.get("fetch_method")) == "snippet_fallback":
            quality = "snippet_only"   # snippets are not full text — not scoreable
        if quality == "ok" and _PRODUCT_STUB_RE.search(_field(r.get("url"))):
            quality = "product_stub"   # Google Shopping stub — not scoreable
        m[k] = {"text": text, "quality": quality}
    src_by_q[q] = m

n_src = sum(len(m) for m in src_by_q.values())
n_bad = sum(1 for m in src_by_q.values() for v in m.values() if v["quality"] != "ok")
print(f"Sources: {n_src} total, {n_bad} excluded (quality != ok).")

# --- 2a. Enumerate statements at Google's NATIVE granularity ---
# A statement = one AIO block unit (paragraph or list item) carrying one
# citation group — exactly how Google renders the overview. Units are NEVER
# split into sentences.

def _norm(s):
    return re.sub(r"\W+", " ", str(s).casefold()).strip()

def _norm_with_map(s):
    """Normalized string + map from each normalized char back to its raw index.
    Lets us locate a normalized substring's span in the RAW text."""
    out, idx_map = [], []
    prev_space = True
    for i, ch in enumerate(s):
        if ch.isalnum():
            out.append(ch.lower()); idx_map.append(i); prev_space = False
        elif not prev_space:
            out.append(" "); idx_map.append(i); prev_space = True
    return "".join(out), idx_map

def _reconstruct_units(aio_text, attributed, min_words=8):
    """Legacy path (datasets without statements_raw.csv): rebuild Google's
    block units from what was saved. Attributed units are used WHOLE; the
    UNCOVERED remainder of aio_text is segmented at bullets/paragraph gaps
    only — one contiguous stretch = one uncited unit."""
    norm_text, idx_map = _norm_with_map(aio_text)
    spans = []          # (raw_start, raw_end, attributed_index)
    unmatched = []
    for ai, e in enumerate(attributed):
        nm = _norm(e["text"])
        pos = norm_text.find(nm) if nm else -1
        if pos >= 0 and nm:
            raw_start = idx_map[pos]
            raw_end   = idx_map[min(pos + len(nm) - 1, len(idx_map) - 1)] + 1
            spans.append((raw_start, raw_end, ai))
        else:
            unmatched.append(ai)
    spans.sort()

    units, cursor = [], 0
    def _emit_uncovered(segment):
        for piece in re.split(r"\s*•\s*|\n{2,}", segment):
            piece = piece.strip(" •:;,.")
            if len(piece.split()) >= min_words:
                units.append({"text": piece, "ids": []})
    for raw_start, raw_end, ai in spans:
        if raw_start > cursor:
            _emit_uncovered(aio_text[cursor:raw_start])
        units.append({"text": attributed[ai]["text"], "ids": attributed[ai]["ids"]})
        cursor = max(cursor, raw_end)
    if cursor < len(aio_text):
        _emit_uncovered(aio_text[cursor:])
    for ai in unmatched:   # attribution never dropped
        units.append({"text": attributed[ai]["text"], "ids": attributed[ai]["ids"]})
    return units

statements_by_q = {}   # q -> ordered list of {"text", "ids"} (ids=[] => uncited)
_stmt_raw_path = os.path.join(OUT_DIR, "statements_raw.csv")

if os.path.exists(_stmt_raw_path):
    # NATIVE path: Step 9 saved Google's block units directly — use verbatim.
    _sr = pd.read_csv(_stmt_raw_path, encoding="utf-8-sig", low_memory=False)
    for q, grp in _sr.groupby("question"):
        stmts = []
        for _, r in grp.sort_values("statement_index").iterrows():
            text = _field(r.get("statement_text"))
            if not text:
                continue
            ids = [k for k in (_src_key(t) for t in
                               re.split(r"[,;\s]+", _field(r.get("citation_indexes"))))
                   if k is not None]
            stmts.append({"text": text, "ids": ids})
        statements_by_q[q] = stmts
    print(f"Statement source: {OUTPUT_SUBDIR}/statements_raw.csv (native block units)")
else:
    # LEGACY path: reconstruct units from supporting_statement columns + aio_text.
    print("Statement source: reconstructed from aio_text (statements_raw.csv not "
          "found — produced by Step 9 on your next pipeline run)")
    for q, grp in df.groupby("question"):
        seen_norm, attributed = set(), []
        for _, r in grp.iterrows():
            for n in range(1, MAX_STATEMENTS + 1):
                stmt = _field(r.get(f"supporting_statement{n}_text"))
                if not stmt or _norm(stmt) in seen_norm:
                    continue
                seen_norm.add(_norm(stmt))
                cits_raw = _field(r.get(f"supporting_statement{n}_citations"))
                ids = [k for k in (_src_key(t) for t in re.split(r"[,:;\s]+", cits_raw))
                       if k is not None]
                attributed.append({"text": stmt, "ids": ids})
        aio_text = _field(grp.iloc[0].get("aio_text"))
        statements_by_q[q] = _reconstruct_units(aio_text, attributed)

n_stmts_all = sum(len(v) for v in statements_by_q.values())
n_cited = sum(1 for v in statements_by_q.values() for s in v if s["ids"])
print(f"Statements enumerated: {n_stmts_all} total "
      f"({n_cited} with Google citations, {n_stmts_all - n_cited} uncited).")

# --- 3. Download the model FILES first (resumable, retries), then load ---
# Separating the download from the load means a stalled shard retries+resumes
# here (with progress) instead of hanging inside from_pretrained. On repeated
# failure with hf_transfer, we disable it and retry with the standard downloader.
_MODEL_ID = "oumi-ai/HallOumi-8B-classifier"   # must match HalloumiScorer's default
from huggingface_hub import snapshot_download
print(f"Fetching model files ({_MODEL_ID}) — ~15 GB, one-time per VM...")
for _attempt in range(1, 6):
    try:
        snapshot_download(_MODEL_ID, resume_download=True, max_workers=4,
                          allow_patterns=["*.safetensors", "*.json", "*.model",
                                          "tokenizer*", "special_tokens*"])
        print("✓ Model files present.")
        break
    except Exception as _e:
        print(f"⚠ download attempt {_attempt}/5 failed: "
              f"{type(_e).__name__}: {str(_e)[:120]}")
        if _attempt == 2 and os.environ.get("HF_HUB_ENABLE_HF_TRANSFER"):
            os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
            print("  → disabling hf_transfer, retrying with the standard downloader...")
        if _attempt == 5:
            print("  → all retries failed; letting the loader attempt once more "
                  "(check your Colab network / try a fresh runtime).")

# --- Load model once (4-bit) ---
print("Loading Halloumi model into GPU (4-bit)...")
halloumi = HalloumiScorer(load_in_4bit=LOAD_IN_4BIT)
if torch.cuda.is_available():
    print(f"4-bit loaded: {getattr(halloumi, 'is_4bit', None)} | "
          f"GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("Halloumi ready.")

# --- 3a. Scoring machinery (refined verdict logic) ---
_pair_cache = {}   # (question, claim, source_idx) -> supported_prob | None

def _score_claim_vs_source(question, claim, src_idx):
    """Max supported-prob of ONE claim/sentence against ONE source (chunked)."""
    key = (question, claim, src_idx)
    if key in _pair_cache:
        return _pair_cache[key]
    entry = src_by_q.get(question, {}).get(src_idx)
    if not entry or entry["quality"] != "ok":
        _pair_cache[key] = None
        return None
    # Stop reading a page once one chunk clears SUPPORT_THRESHOLD (user decision
    # 2026-09-19, for speed). Every verdict is identical to a full scan -- only
    # the threshold is ever compared -- but supported_prob / mean_prob for a
    # supported sentence become "first chunk >= threshold" rather than the page max.
    out = halloumi.score_max(entry["text"], claim, chunk_tokens=CHUNK_TOKENS,
                             stop_at=SUPPORT_THRESHOLD)
    _pair_cache[key] = out["supported_prob"]
    return out["supported_prob"]

# ── Sentence-level claim handling ─────────────────────────────────────────────
# HallOumi is a per-SENTENCE entailment model. Feeding it a whole merged
# paragraph as one "claim" scored well-grounded facts as low (e.g. the DASH
# statement scored 0.19 against the Mayo page that contains DASH guidance,
# because the paragraph also held a non-factual directive). So we split each
# paragraph statement into sentences, drop non-factual/navigational ones, score
# each factual sentence against ANY of the cited sources, and call the paragraph
# supported when a majority of its factual sentences are supported.

# A sentence is NON-FACTUAL (skip it) when it is a lead-in, a navigational
# directive, a bare URL, or an AI/meta disclaimer — none are verifiable claims.
_DIRECTIVE_START_RE = re.compile(
    r"^\s*(follow|use|explore|visit|see|read|refer|consult|check\s+out|"
    r"learn\s+more|find|discover|review|browse|download|click|watch|"
    r"for\s+(more|granular|detailed|structured|specific|comprehensive|"
    r"additional|further))\b", re.I)
_DIRECTIVE_ANY_RE = re.compile(
    r"(https?://|\bwww\.|\bi am an ai\b|not a doctor|"
    r"consult (a|an|your|with)? ?(healthcare|health care|doctor|physician|"
    r"medical|professional)|could you share|i can (give|provide|offer)|"
    r"let me know|tailored to your|for (structured|specific) (meal )?planning|"
    r"for (more|further) (details|information|guidance|reading))", re.I)

def _is_factual_sentence(sent):
    s = sent.strip()
    if len(s.split()) < 4:
        return False
    if s.endswith(":"):
        return False   # lead-in ("...incorporate these changes:")
    if _DIRECTIVE_START_RE.search(s):
        return False
    if _DIRECTIVE_ANY_RE.search(s):
        return False
    return True

def _factual_sentences(statement):
    """Split a paragraph statement into factual sentence-claims."""
    return [s for s in halloumi._split_into_claims(statement)
            if _is_factual_sentence(s)]

# ── Bullet-label stripping & enumeration decomposition ───────────────────────
# Two scoring artifacts found by manual audit of the 7/11 run:
#   1. "Healthy Fats: olive oil, avocados..." — the "Label:" prefix isn't a
#      claim, and it drags the entailment score down (confirmed false negative
#      against a Mayo page that contains every listed fact).
#   2. Conjunction lists score all-or-nothing: "spinach, kale, broccoli,
#      blueberries, and oranges provide X" fails when the source omits ONE item.
# Fix: strip the label before scoring; if the whole sentence still fails,
# decompose the enumeration into per-item sub-claims and apply majority rule.
_LABEL_PREFIX_RE = re.compile(r"^[A-Z][\w&' \-]{0,40}:\s+")
# Subject enumeration at the START of a claim followed by a SHARED predicate:
# "A, B, C, and D <verb ...>". Anchoring on the leading subject list + a predicate
# verb keeps the last item from swallowing the predicate — the earlier greedy
# regex turned "Spinach, kale... provide antioxidants, vitamins, and potassium"
# into garbage like "Spinachs, vitamins, and potassium".
_ENUM_SUBJECT_RE = re.compile(
    r"^(?P<subj>(?:[\w\-'()]+(?: [\w\-'()]+){0,3},\s+){2,}"
    r"(?:and\s+|or\s+)?[\w\-'()]+(?: [\w\-'()]+){0,3})\s+"
    r"(?P<pred>(?:provide|provides|are|is|deliver|delivers|contain|contains|"
    r"include|includes|reduce|reduces|help|helps|offer|offers|lower|lowers|"
    r"boost|boosts|keep|keeps|support|supports|give|gives|add|adds|"
    r"promote|promotes|improve|improves)\b.*)$", re.I)

def _strip_label(sent):
    s = re.sub(r"^[•*\-\s]+", "", str(sent).strip())
    return _LABEL_PREFIX_RE.sub("", s)

def _decompose_enumeration(sent):
    """'A, B, C, and D <predicate>' -> one sub-claim per SUBJECT item, each
    keeping the shared predicate: 'A <predicate>'. Requires the list to sit at
    the start of the claim and be followed by a predicate verb, so the last item
    can't absorb the predicate. [] when no >=3-item subject enumeration exists."""
    m = _ENUM_SUBJECT_RE.match(sent.strip())
    if not m:
        return []
    pred = m.group("pred").strip()
    items = [i.strip() for i in
             re.split(r",\s*(?:and\s+|or\s+)?|\s+and\s+|\s+or\s+", m.group("subj"))
             if i.strip()]
    if len(items) < 3:
        return []
    return [f"{item} {pred}" for item in items]

def _sentence_support(question, sent, sources):
    """(supported, best_prob|None, best_source|None) for ONE sentence.
    Stage 1: score the label-stripped sentence whole against each source.
    Stage 2 (only when stage 1 misses the threshold): decompose enumerations
    into per-item sub-claims; supported when >= SUPPORT_FRACTION of items are.
    Stage 2 only runs on failures, so the extra GPU cost stays small."""
    claim = _strip_label(sent)
    best_p, best_c = None, None
    for c in sources:
        p = _score_claim_vs_source(question, claim, c)
        if p is not None and (best_p is None or p > best_p):
            best_p, best_c = p, c
    if best_p is not None and best_p >= SUPPORT_THRESHOLD:
        return True, best_p, best_c
    subs = _decompose_enumeration(claim)
    if best_p is not None and len(subs) >= 3:
        n_sub_ok = 0
        for sub in subs:
            sub_p = None
            for c in sources:
                p = _score_claim_vs_source(question, sub, c)
                if p is not None and (sub_p is None or p > sub_p):
                    sub_p = p
                    if p > best_p:
                        best_p, best_c = p, c
            if sub_p is not None and sub_p >= SUPPORT_THRESHOLD:
                n_sub_ok += 1
        if n_sub_ok / len(subs) >= SUPPORT_FRACTION:
            return True, best_p, best_c
    return False, best_p, best_c

# ── Uncited-statement sub-classification ─────────────────────────────────────
# "uncited" mixes three species; only the last matters for hallucination math:
#   header          — section label ("Best Foods to Include")
#   conversational  — Gemini talking to the user ("could you let me know:")
#   uncited_factual — a real claim Google attached no citation to
_CONVERSATIONAL_RE = re.compile(
    r"^\s*(if you|do you|could you|would you|tell me|what is|which of)|"
    r"\bI can\b[^.!?]{0,25}?\b(help|give|provide|offer|build|create|calculate|"
    r"tailor)\b|\blet me know\b|\byour specific\b|\btelling me\b|"
    r"\bshare your\b", re.I)

# Referral / personalisation directives — "go ask someone else", not a claim any
# source could ground. Deliberately scoped to _uncited_type: _DIRECTIVE_ANY_RE
# also gates _is_factual_sentence, so widening THAT would change Halloumi
# verdicts, not just the uncited labels.
_REFERRAL_RE = re.compile(
    r"^\s*(for (a )?personalized|for your (own|specific))|"
    r"^\s*check\b|"
    r"\bconsult\s+(?:\w+\s+){0,5}?(healthcare|health care|doctor|physician|"
    r"medical|professional)", re.I)

def _is_conversational_text(t):
    return bool("?" in t or _CONVERSATIONAL_RE.search(t)
                or _DIRECTIVE_ANY_RE.search(t) or _REFERRAL_RE.search(t))

def _uncited_type(text, prev_text=""):
    """header | conversational | uncited_factual.

    prev_text is the PRECEDING statement of the same overview. Gemini's
    ask-the-user blocks span two units — "...I can help you calculate it by
    telling me:" followed by the answer slots ("Your skin type | Your
    location"). The slot list carries no conversational cue of its own, so
    judged alone it reads as a factual claim and wrongly inflates the
    uncited_factual denominator of the strict support rate.
    """
    t = str(text).strip()
    if _is_conversational_text(t):
        return "conversational"
    p = str(prev_text).strip()
    if p.endswith(":") and _is_conversational_text(p):
        return "conversational"   # answer slots for the question just asked
    if t.endswith(":") or (len(t.split()) <= 6 and not re.search(r"[.!?]$", t)):
        return "header"
    return "uncited_factual"

def _any_source_verdict(question, statement):
    """Majority-rule verdict for a statement against ALL readable sources of
    its question (not just the cited ones). Second opinion on 'unsupported'
    statements: separates "Google cited the wrong source" from "not grounded
    anywhere we can see". -> (supported_bool|None, best_source|None)"""
    srcs = [c for c, v in src_by_q.get(question, {}).items()
            if v["quality"] == "ok"]
    sentences = _factual_sentences(statement)
    if not sentences or not srcs:
        return None, None
    n_ok, best = 0, (None, None)
    for sent in sentences:
        ok, bp, bc = _sentence_support(question, sent, srcs)
        if ok:
            n_ok += 1
        if bp is not None and (best[0] is None or bp > best[0]):
            best = (bp, bc)
    return (n_ok / len(sentences)) >= SUPPORT_FRACTION, best[1]

def _verdict(question, statement, citation_ids):
    """Sentence-level, ANY-source, majority-rule verdict.
      • each factual sentence is SUPPORTED if ANY readable cited source's
        supported-prob >= threshold (any relevant fact in any source counts)
      • paragraph SUPPORTED if >=50% of its factual sentences are supported
      • else SKIPPED (indeterminate) if a cited source was unreadable
      • else UNSUPPORTED (all sources readable, majority not met)
    """
    srcmap = src_by_q.get(question, {})
    readable   = [c for c in citation_ids
                  if srcmap.get(c) and srcmap[c]["quality"] == "ok"]
    unreadable = [c for c in citation_ids if c not in readable]

    sentences = _factual_sentences(statement)
    n_sent_all = len(halloumi._split_into_claims(statement))

    rec = {
        "cited_source_indexes":      ",".join(str(c) for c in citation_ids),
        "unreadable_source_indexes": ",".join(str(c) for c in sorted(unreadable)),
        "n_sentences":  n_sent_all,
        "n_factual":    len(sentences),
        "n_supported_sentences": 0,
        "support_fraction": None,
        "supported_prob": None,   # best factual-sentence max, for reference
        "best_source":    None,
        "mean_prob":      None,
        "skip_reason":    "",
        "uncited_type":   "",
        "supported_by_any_source": None,
        "any_source_best": None,
    }

    # No verifiable claims after filtering — can't judge support
    if not sentences:
        rec["status"] = "skipped"
        rec["skip_reason"] = "no_factual_claims"
        return rec

    n_supported = 0
    best_overall = (-1.0, None)   # (prob, source) for the single best sentence hit
    sentence_maxes = []
    for sent in sentences:
        ok, bp, bc = _sentence_support(question, sent, readable)
        if bp is not None:
            sentence_maxes.append(bp)
            if bp > best_overall[0]:
                best_overall = (bp, bc)
        if ok:
            n_supported += 1

    frac = n_supported / len(sentences)
    rec["n_supported_sentences"] = n_supported
    rec["support_fraction"]      = round(frac, 3)
    if best_overall[1] is not None:
        rec["supported_prob"] = best_overall[0]
        rec["best_source"]    = best_overall[1]
    if sentence_maxes:
        rec["mean_prob"] = round(sum(sentence_maxes) / len(sentence_maxes), 4)

    if frac >= SUPPORT_FRACTION:
        rec["status"] = "supported"
    elif unreadable:
        rec["status"] = "skipped"
        rec["skip_reason"] = ("indeterminate: unreadable_source(s): "
                              + ",".join(str(c) for c in sorted(unreadable)))
    else:
        rec["status"] = "unsupported"
    return rec

# --- 3b. Smoke test: ONE forward set so an OOM surfaces in seconds ---
def _smoke_test():
    import time as _time
    for q, stmts in statements_by_q.items():
        for s in stmts:
            readable = [c for c in s["ids"]
                        if src_by_q.get(q, {}).get(c, {}).get("quality") == "ok"]
            if readable:
                t0 = _time.time()
                p = _score_claim_vs_source(q, s["text"], readable[0])
                return ("statement-vs-source", p, _time.time() - t0)
    return (None, None, 0)

print("Running smoke test (1 forward set)...")
_mode, _p, _secs = _smoke_test()
if _mode is None:
    print("⚠️  Smoke test found nothing scorable — no cited statement has a readable source.")
else:
    print(f"✓ Smoke test OK ({_mode}, {_secs:.1f}s): supported={_p}")
    if torch.cuda.is_available():
        print(f"  GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB "
              f"(reserved {torch.cuda.memory_reserved()/1e9:.2f} GB)")

# --- 4. Score every enumerated statement ---
# Progress is per STATEMENT (not per question) so a slow statement can't make
# the run look frozen. Each statement that takes >8s prints its time + source
# sizes, so a genuine stall is visible instead of a silent hang.
import time as _time
stmt_rows = []              # statement_scores.csv rows
verdict_by_norm = {}        # (q, norm) -> record, for per-source column lookup
q_totals = {q: {"total": len(v), "supported": 0, "unsupported": 0,
                "skipped": 0, "uncited": 0, "uncited_header": 0,
                "uncited_conversational": 0, "uncited_factual": 0}
            for q, v in statements_by_q.items()}

_flat = [(q, i, s) for q in sorted(statements_by_q)
         for i, s in enumerate(statements_by_q[q], 1)]
_run_t0 = _time.time()

def _ckpt_key(q, i, text):
    # verdict parameters are part of the key: changing them rescores everything
    raw = "\x1f".join([q, str(i), text, str(SUPPORT_THRESHOLD), str(SUPPORT_FRACTION)])
    return hashlib.md5(raw.encode("utf-8")).hexdigest()

_done = {}
if os.path.exists(CKPT_FILE):
    with open(CKPT_FILE, encoding="utf-8") as _f:
        for _line in _f:
            try:
                _r = _json.loads(_line)
                _done[_r["key"]] = _r["rec"]
            except Exception:
                pass   # a half-written last line from a killed run
_n_cited_all = sum(1 for _, _, s in _flat if s["ids"])
_n_cited_done = sum(1 for q, i, s in _flat if s["ids"] and _ckpt_key(q, i, s["text"]) in _done)
print(f"HALLOUMI checkpoint: {_n_cited_done}/{_n_cited_all} cited statements already scored "
      f"({len(_done)} checkpoint rows)")
if MAX_HOURS:
    print(f"Will pause on its own after {MAX_HOURS:g} h.")
_ckpt_fh = open(CKPT_FILE, "a", encoding="utf-8")
_sess_t0, _sess_n, _paused = _time.time(), 0, None

for q, i, s in tqdm(_flat, desc="Statements", unit="stmt", mininterval=120):
    _key = _ckpt_key(q, i, s["text"])
    if _key in _done:
        rec = _done[_key]
    elif os.path.exists(STOP_FLAG):
        _paused = "pause flag found"
        break
    elif MAX_HOURS and _time.time() - _sess_t0 > MAX_HOURS * 3600:
        _paused = f"{MAX_HOURS:g}-hour limit reached"
        break
    else:
        rec = None
    _s0 = _time.time()
    if rec is not None:
        pass
    elif not s["ids"]:
        rec = {"status": "uncited", "cited_source_indexes": "",
               "unreadable_source_indexes": "", "n_sentences": 0,
               "n_factual": 0, "n_supported_sentences": 0,
               "support_fraction": None, "supported_prob": None,
               "best_source": None, "mean_prob": None, "skip_reason": "",
               "uncited_type": _uncited_type(
                   s["text"],
                   statements_by_q[q][i - 2]["text"] if i > 1 else ""),
               "supported_by_any_source": None, "any_source_best": None}
    else:
        rec = _verdict(q, s["text"], s["ids"])
        # Second opinion DISABLED (user decision 2026-09-19): re-checking every
        # unsupported statement against ALL of the question's sources read each
        # page in full again and roughly doubled the run. The question here is
        # whether the statement is supported by the source GOOGLE cited, which
        # _verdict already answers. supported_by_any_source / any_source_best
        # stay empty from here on (they are filled for the first ~600 cited
        # statements, scored before this change). _any_source_verdict is kept
        # above so the check can be re-enabled by restoring this call.
    if _key not in _done:
        _done[_key] = rec
        _ckpt_fh.write(_json.dumps({"key": _key, "rec": rec}, default=str) + "\n")
        _ckpt_fh.flush()
        if s["ids"]:
            _n_cited_done += 1
            _sess_n += 1
            if _sess_n % 50 == 0:
                _rate = _sess_n / max(_time.time() - _sess_t0, 1) * 3600
                _left = (_n_cited_all - _n_cited_done) / max(_rate, 1e-9)
                print(f"HALLOUMI checkpoint: {_n_cited_done}/{_n_cited_all} cited statements "
                      f"({_rate:.0f}/h, ~{_left:.1f} h of GPU time left)", flush=True)
    _dt = _time.time() - _s0
    if _dt > 8:   # surface slow statements so slow != stuck
        readable = [c for c in s["ids"]
                    if src_by_q.get(q, {}).get(c, {}).get("quality") == "ok"]
        sizes = "+".join(str(len(src_by_q[q][c]["text"]) // 1000) + "k"
                         for c in readable if c in src_by_q.get(q, {}))
        print(f"  ⏱ {_dt:.0f}s — stmt {i} of '{q[:30]}' "
              f"({len(readable)} src: {sizes})")
    t = q_totals[q]
    t[rec["status"]] += 1
    if rec["status"] == "uncited":
        # uncited_type is header|conversational|uncited_factual; only the
        # first two need the uncited_ counter prefix added.
        _ut = rec["uncited_type"]
        t[_ut if _ut.startswith("uncited_") else "uncited_" + _ut] += 1
    verdict_by_norm[(q, _norm(s["text"]))] = rec
    stmt_rows.append({
        "question":        q,
        "statement_index": i,
        "statement_text":  s["text"],
        "status":          rec["status"],
        "cited_source_indexes":      rec["cited_source_indexes"],
        "unreadable_source_indexes": rec["unreadable_source_indexes"],
        "n_sentences":            rec["n_sentences"],
        "n_factual":              rec["n_factual"],
        "n_supported_sentences":  rec["n_supported_sentences"],
        "support_fraction":       rec["support_fraction"],
        "supported_prob":  rec["supported_prob"],
        "best_source":     rec["best_source"],
        "mean_prob":       rec["mean_prob"],
        "skip_reason":     rec["skip_reason"],
        "uncited_type":    rec["uncited_type"],
        "supported_by_any_source": rec["supported_by_any_source"],
        "any_source_best": rec["any_source_best"],
        "threshold":       SUPPORT_THRESHOLD,
    })

_ckpt_fh.close()
if _paused:
    print(f"\nHALLOUMI PAUSED ({_paused}): {_n_cited_done}/{_n_cited_all} cited statements scored; "
          f"{_sess_n} this session. Progress is saved in {os.path.basename(CKPT_FILE)}; "
          "run halloumi_ctl.py start to resume.", flush=True)
    raise SystemExit(3)

for t in q_totals.values():
    t["scored"] = t["supported"] + t["unsupported"]
print(f"Scored {len(_flat)} statements in {_time.time() - _run_t0:.0f}s.")

# --- 4b. PER-AIO statement columns (identical across a question's rows) ---
# Every row of a question gets the SAME aio_statement{N}_* columns: statement 1
# is always the first block unit of the overview, so the columns line up down
# the rows. Legacy per-source supporting_statement* columns are dropped.
MAX_AIO_STMT_COLS = 15
stmt_cols_by_q = {}   # q -> {colname: value}
for q, stmts in statements_by_q.items():
    cols = {}
    for n, s in enumerate(stmts, 1):
        if n > MAX_AIO_STMT_COLS:
            break
        rec = verdict_by_norm.get((q, _norm(s["text"])), {})
        cols[f"aio_statement{n}_text"]      = s["text"]
        cols[f"aio_statement{n}_citations"] = ",".join(str(c) for c in s["ids"])
        cols[f"aio_statement{n}_status"]    = rec.get("status", "")
        cols[f"aio_statement{n}_supported"] = (
            True if rec.get("status") == "supported"
            else (False if rec.get("status") == "unsupported" else None))
        cols[f"aio_statement{n}_support_fraction"] = rec.get("support_fraction")
        cols[f"aio_statement{n}_best_source"]      = rec.get("best_source")
    stmt_cols_by_q[q] = cols

# Drop any legacy per-source supporting_statement* columns so old datasets get
# cleaned up even if Step 9 wasn't re-run.
_legacy = [c for c in df.columns if c.startswith("supporting_statement")]
records = []
for _, row in df.iterrows():
    out = row.to_dict()
    for c in _legacy:
        out.pop(c, None)
    q = row.get("question")
    out.update(stmt_cols_by_q.get(q, {}))
    t = q_totals.get(q)
    if t:
        out["aio_halloumi_n_statements_total"]     = t["total"]
        out["aio_halloumi_n_statements_supported"] = t["supported"]
        out["aio_halloumi_n_statements_scored"]    = t["scored"]
        out["aio_halloumi_n_statements_skipped"]   = t["skipped"]
        out["aio_halloumi_n_statements_uncited"]   = t["uncited"]
        out["aio_halloumi_support_score"]          = (f"{t['supported']}/{t['scored']}"
                                                      if t["scored"] else "")
        out["aio_halloumi_support_rate"]           = (round(t["supported"] / t["scored"], 3)
                                                      if t["scored"] else None)
        out["aio_halloumi_n_statements_uncited_factual"] = t["uncited_factual"]
        _strict_den = t["scored"] + t["uncited_factual"]
        out["aio_halloumi_support_rate_strict"]    = (round(t["supported"] / _strict_den, 3)
                                                      if _strict_den else None)
    records.append(out)

# --- 4c. Run summary ---
print(f"\n=== RUN SUMMARY (sentence threshold {SUPPORT_THRESHOLD}, "
      f"statement needs >={SUPPORT_FRACTION:.0%} of factual sentences) ===")
print(f"{'question':<44} {'score':>7} {'strict':>8} {'total':>6} {'scored':>7} {'skipped':>8} {'uncited':>8}")
grand = {"total": 0, "supported": 0, "scored": 0, "skipped": 0, "uncited": 0,
         "uncited_factual": 0}
for q in sorted(q_totals):
    t = q_totals[q]
    score = f"{t['supported']}/{t['scored']}" if t["scored"] else "—"
    _sd = t["scored"] + t["uncited_factual"]
    strict = f"{t['supported']}/{_sd}" if _sd else "—"
    print(f"{q[:42]:<44} {score:>7} {strict:>8} {t['total']:>6} {t['scored']:>7} "
          f"{t['skipped']:>8} {t['uncited']:>8}")
    for k in grand:
        grand[k] += t[k]
_gsd = grand['scored'] + grand['uncited_factual']
print(f"{'TOTAL':<44} {str(grand['supported']) + '/' + str(grand['scored']):>7} "
      f"{str(grand['supported']) + '/' + str(_gsd):>8} "
      f"{grand['total']:>6} {grand['scored']:>7} {grand['skipped']:>8} {grand['uncited']:>8}")
print(f"Excluded sources (bad text): {n_bad}/{n_src} | forward pairs: {len(_pair_cache)}")

# --- 5. Save (everything in Output/) ---
pd.DataFrame(stmt_rows).to_csv(os.path.join(OUT_DIR, STMT_FILE), index=False,
                               encoding="utf-8-sig", quoting=csv.QUOTE_ALL)
print(f"\n💾 {len(stmt_rows)} statement scores → {OUTPUT_SUBDIR}/{STMT_FILE}")

out_df = pd.DataFrame(records)
out_df.to_csv(path, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_ALL)
print(f"💾 Enriched {len(out_df)} rows in place → {OUTPUT_SUBDIR}/{AIO_FILE}")

# --- 6. Rebuild comparison CSVs + THE workbook (with all scores in it) ---
print("\nRebuilding comparison CSVs + analysis workbook...")
import comparison_builder
importlib.reload(comparison_builder)
summary = comparison_builder.build_comparison_outputs(
    PROJECT_DIR, output_subdir=OUTPUT_SUBDIR, aio_file=AIO_FILE,
    stmt_file=STMT_FILE, include_workbook=True)
print(f"\n✅ Halloumi scoring complete — all outputs in {summary['output_dir']}")
